# Modèle lourd : DistilBERT mesuré par CodeCarbon (Kaggle) · Ons

Chaîne complète : données (code commun) → DistilBERT, 1 époque → accuracy + `emissions.csv`.

Notebook **autonome** : le code du dépôt (`src/commun.py`, `src/modele_lourd.py`) est recopié
ci-dessous, aucun jeton GitHub n'est nécessaire.

**Avant *Run All*** (panneau de droite) :
1. *Session options → Accelerator* : **GPU T4 x2** (ou GPU P100).
2. *Internet* : **On** (compte Kaggle vérifié par téléphone).

Séance 2 : seul le **test court** est lancé. Le vrai run (section 3) est désactivé jusqu'à la séance 3.

In [ ]:
!nvidia-smi

Un seul GPU : sinon le Trainer répartit sur les deux T4 et la mesure n'est plus comparable.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.makedirs("src", exist_ok=True)
open("src/__init__.py", "w").close()

In [ ]:
!pip install -q codecarbon

## 1. Code du dépôt (copie de `src/commun.py` et `src/modele_lourd.py`)

In [ ]:
%%writefile src/commun.py
"""Code commun : identique pour tous (mêmes données, même mesure).

Copié tel quel de la fiche de répartition (séance 2). Ne pas modifier seul :
toute modification se décide en équipe, sinon les mesures ne sont plus comparables.
"""

from datasets import load_dataset
from codecarbon import EmissionsTracker

SEED = 42


def load_data(n_train=10_000, n_test=2_000, seed=SEED):
    # "ag_news" seul est refusé par huggingface_hub récent (id « namespace/name » exigé) :
    # fancyzhx/ag_news est le même jeu de données sous son identifiant actuel.
    ds = load_dataset("fancyzhx/ag_news")
    train = ds["train"].shuffle(seed=seed).select(range(n_train))
    test = ds["test"].shuffle(seed=seed).select(range(n_test))
    return train, test  # colonnes : "text", "label" (4 classes)


def mesurer(nom, fonction_entrainement):
    tracker = EmissionsTracker(project_name=nom, output_file="emissions.csv")
    tracker.start()
    accuracy = fonction_entrainement()
    kg = tracker.stop()
    print(f"{nom} | accuracy={accuracy:.4f} | {kg*1000:.3f} g CO2eq")
    return accuracy, kg


# Ali : mesurer("leger", train_light)    Ons : mesurer("lourd", train_heavy)

In [ ]:
%%writefile src/modele_lourd.py
"""Modèle lourd : fine-tuning de DistilBERT, 1 époque (responsable : Ons).

Chaîne complète : données (code commun) -> DistilBERT (HF Trainer) -> accuracy
-> emissions.csv (CodeCarbon, via mesurer()).

À lancer sur Colab avec un GPU T4 (notebooks/lourd_distilbert_colab.ipynb), ou :

    python -m src.modele_lourd --test-court     # ~500 exemples : valide la mesure
    python -m src.modele_lourd                  # vrai run : 10 000 / 2 000

Chaque run écrit aussi resultats/<nom>.json : accuracy, durée, GPU, hyperparamètres,
et l'estimation de durée du vrai run (pour vérifier la contrainte < 20 min).
"""

from __future__ import annotations

import argparse
import json
import platform
import time
from pathlib import Path
from typing import Any

from src.commun import SEED, load_data, mesurer

# Hyperparamètres du modèle lourd (à reporter dans docs/modele_lourd.md).
HEAVY_PARAMS: dict[str, Any] = {
    "model_name": "distilbert-base-uncased",
    "num_labels": 4,
    "epochs": 1,
    "batch_size": 16,
    "learning_rate": 5e-5,
    "weight_decay": 0.01,
    "max_length": 128,
}

N_TRAIN_VRAI_RUN = 10_000
LIMITE_MINUTES = 20


def infos_materiel() -> dict[str, Any]:
    """Décrit le matériel utilisé (pour le document « Hypothèses »)."""
    import torch

    infos: dict[str, Any] = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "cpu": platform.processor() or platform.machine(),
        "torch": torch.__version__,
    }
    if torch.cuda.is_available():
        infos["gpu"] = torch.cuda.get_device_name(0)
    return infos


def train_heavy(
    train: Any,
    test: Any,
    n_train: int | None = None,
    n_test: int | None = None,
    params: dict[str, Any] | None = None,
    nom: str = "lourd",
    dossier_resultats: str = "resultats",
) -> float:
    """Fine-tune DistilBERT sur `train`, renvoie l'accuracy sur `test`.

    `train` et `test` sont les datasets renvoyés par load_data() (colonnes text, label).
    n_train / n_test limitent le nombre d'exemples (test court).
    """
    import torch
    import transformers
    from sklearn.metrics import accuracy_score
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        set_seed,
    )

    p = {**HEAVY_PARAMS, **(params or {})}
    set_seed(SEED)

    if n_train is not None:
        train = train.select(range(min(n_train, len(train))))
    if n_test is not None:
        test = test.select(range(min(n_test, len(test))))

    tokenizer = AutoTokenizer.from_pretrained(p["model_name"])

    def encoder(batch: dict[str, list]) -> dict[str, list]:
        return tokenizer(batch["text"], truncation=True, max_length=p["max_length"])

    ds_train = train.map(encoder, batched=True, remove_columns=["text"])
    ds_test = test.map(encoder, batched=True, remove_columns=["text"])

    model = AutoModelForSequenceClassification.from_pretrained(
        p["model_name"], num_labels=p["num_labels"]
    )
    args = TrainingArguments(
        output_dir=f"outputs/{nom}",
        num_train_epochs=p["epochs"],
        per_device_train_batch_size=p["batch_size"],
        per_device_eval_batch_size=p["batch_size"] * 2,
        learning_rate=p["learning_rate"],
        weight_decay=p["weight_decay"],
        seed=SEED,
        fp16=torch.cuda.is_available(),
        save_strategy="no",
        report_to="none",
        logging_steps=50,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_train,
        data_collator=DataCollatorWithPadding(tokenizer),
    )

    debut = time.perf_counter()
    trainer.train()
    duree_train = time.perf_counter() - debut

    preds = trainer.predict(ds_test).predictions.argmax(axis=-1)
    accuracy = float(accuracy_score(ds_test["label"], preds))

    # Estimation linéaire de la durée d'entraînement du vrai run (10 000 exemples).
    estimation_min = duree_train / len(ds_train) * N_TRAIN_VRAI_RUN * p["epochs"] / 60
    infos = {
        "nom": nom,
        "accuracy": accuracy,
        "n_train": len(ds_train),
        "n_test": len(ds_test),
        "duree_entrainement_s": round(duree_train, 1),
        "estimation_vrai_run_min": round(estimation_min, 1),
        "respecte_limite_20_min": estimation_min < LIMITE_MINUTES,
        "hyperparametres": p,
        "materiel": infos_materiel(),
        "transformers": transformers.__version__,
    }
    Path(dossier_resultats).mkdir(exist_ok=True)
    chemin = Path(dossier_resultats) / f"{nom}.json"
    chemin.write_text(json.dumps(infos, indent=2, ensure_ascii=False), encoding="utf-8")
    print(json.dumps(infos, indent=2, ensure_ascii=False))
    return accuracy


def main() -> None:
    parser = argparse.ArgumentParser(description="Modèle lourd mesuré par CodeCarbon")
    parser.add_argument(
        "--test-court", action="store_true", help="500 exemples train / 200 test"
    )
    args = parser.parse_args()

    train, test = load_data()
    if args.test_court:
        nom, n_train, n_test = "lourd_test_court", 500, 200
    else:
        nom, n_train, n_test = "lourd", None, None

    mesurer(nom, lambda: train_heavy(train, test, n_train, n_test, nom=nom))


if __name__ == "__main__":
    main()

## 2. Test court (500 exemples train / 200 test) : valider la mesure, estimer la durée du vrai run

In [ ]:
!python -m src.modele_lourd --test-court

In [ ]:
import json
import pandas as pd

res = json.load(open("resultats/lourd_test_court.json"))
em = pd.read_csv("emissions.csv").iloc[-1]
print("GPU                      :", res["materiel"].get("gpu", res["materiel"]["device"]))
print("Accuracy                 :", round(res["accuracy"], 4))
print("Durée entraînement (s)   :", res["duree_entrainement_s"])
print("Énergie (kWh)            :", em["energy_consumed"])
print("Émissions (g CO2eq)      :", round(em["emissions"] * 1000, 3))
print("Estimation vrai run (min):", res["estimation_vrai_run_min"], "| < 20 min :", res["respecte_limite_20_min"])
pd.read_csv("emissions.csv")

## 3. Vrai run (séance 3) : 10 000 / 2 000

À lancer seulement si l'estimation ci-dessus est < 20 min : passer `LANCER_VRAI_RUN` à `True`.

In [ ]:
LANCER_VRAI_RUN = False
if LANCER_VRAI_RUN:
    !python -m src.modele_lourd

## 4. Récupérer les résultats

*Save Version* (en haut à droite), puis onglet *Output* : télécharger `emissions.csv` et
`resultats/*.json`, et les ajouter au dépôt dans `resultats/kaggle/`.